In [ ]:

import pandas as pd

df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])

# Renombrar columnas
df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Columna 'Canción' ya renombrada
df = df.drop(columns=['Canción'])
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]
df_small = df_filtered[df_filtered['Año'].between(2000, 2010)]


df.head()




,Año,Artista,Máxima_posición
0,2000,Santana Featuring Rob Thomas,1
1,2000,Brian McKnight,2
2,2000,Jessica Simpson,3
3,2000,Whitney Houston,4
4,2000,"Missy ""Misdemeanor"" Elliott Featuring NAS| EVE...",5


In [ ]:
import pandas as pd
import altair as alt
from google.colab import files  # Para subir archivos si no existe el CSV

# --- Cargar archivo ---
try:
    df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')
except FileNotFoundError:
    print("Error: El archivo 'BDD_Billboard_copia_Limpia_2000.csv' no fue encontrado.")
    print("Por favor, súbelo con:")
    print("from google.colab import files; uploaded = files.upload()")
    raise

# --- Preparar datos ---
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])

df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

df = df.drop(columns=['Canción'])
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]
df_small = df_filtered[df_filtered['Año'].between(2000, 2010)]

pivot_table = df_small.pivot_table(
    index='Artista',
    columns='Año',
    values='Máxima_posición',
    aggfunc='count',
    fill_value=0
)

# Top 20 artistas con más canciones en el top 10
top_artistas = pivot_table.sum(axis=1).sort_values(ascending=False).head(20).index
pivot_table_top = pivot_table.loc[top_artistas]

# --- Transformar datos a formato largo ---
df_long = pivot_table_top.reset_index().melt(
    id_vars='Artista',
    var_name='Año',
    value_name='Cantidad'
)

df_long['Año'] = df_long['Año'].astype(str)

# --- Crear selecciones interactivas ---
brush = alt.selection_interval(encodings=['x', 'y'])
highlight = alt.selection_point(fields=['Artista'], empty='none')

# --- Heatmap en tonos morados ---
heatmap = (
    alt.Chart(df_long)
    .mark_rect()
    .encode(
        x=alt.X('Año:O', title='Año'),
        y=alt.Y(
            'Artista:O',
            title='Artista',
            sort=alt.EncodingSortField(field='Cantidad', order='descending')
        ),
        color=alt.condition(
            alt.datum.Cantidad > 0,
            alt.Color('Cantidad:Q',
                      scale=alt.Scale(scheme='purples'),
                      title='Cantidad de canciones'),
            alt.value('white')  # Deja en blanco los espacios sin valor
        ),
        tooltip=['Artista', 'Año', 'Cantidad']
    )
    .properties(
        width=700,
        height=500,
        title='Cantidad de canciones en top 10 por artista y año (2000–2010)'
    )
    .add_params(brush, highlight)
    .transform_filter(brush)
)

# --- Texto dinámico al resaltar artista ---
text = (
    alt.Chart(df_long)
    .mark_text(
        align='left',
        baseline='middle',
        dx=5,
        dy=-5,
        fontWeight='bold',
        color='black'
    )
    .encode(
        x='Año:O',
        y='Artista:O',
        text=alt.condition(highlight, 'Cantidad:Q', alt.value(''))
    )
    .add_params(highlight)
)

# --- Combinar y aplicar estilo con Poppins ---
chart = (heatmap + text).configure(
    title=alt.TitleConfig(font='Poppins', fontSize=18, anchor='start', color='#4527a0'),
    axis=alt.AxisConfig(labelFont='Poppins', titleFont='Poppins', labelFontSize=12, titleFontSize=13),
    legend=alt.LegendConfig(labelFont='Poppins', titleFont='Poppins'),
    view=alt.ViewConfig(stroke='transparent'),  # sin borde
    background='white'  # fondo blanco limpio
)

chart


alt.LayerChart(...)

In [ ]:
import pandas as pd


df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')

# Convertir columna 'date' a datetime y extraer solo el año
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

# Eliminar columnas
df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])


df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Eliminar la columna 'Canción'
df = df.drop(columns=['Canción'])

# Filtrar para máximo posición entre 1 y 10
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]

# Dividir en dos dataframes por rangos de años
df_2000_2010 = df_filtered[df_filtered['Año'].between(2000, 2010)]
df_2011_2025 = df_filtered[df_filtered['Año'].between(2011, 2025)]


print("\nDatos 2011-2025:")
print(df_2011_2025.head())


Datos 2011-2025:
        Año                  Artista  Máxima_posición
57400  2011               Bruno Mars                1
57401  2011               Katy Perry                1
57402  2011                    Ke$ha                1
57403  2011  Rihanna Featuring Drake                1
57404  2011                     P!nk                1


In [ ]:
import pandas as pd


df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')

# Convertir columna 'date' a datetime y extraer solo el año
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

# Eliminar columnas
df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])


df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Eliminar la columna 'Canción'
df = df.drop(columns=['Canción'])

# Filtrar para máximo posición entre 1 y 10
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]


df_2011_2025 = df_filtered[df_filtered['Año'].between(2011, 2025)]


print("\nDatos 2011-2025:")
print(df_2011_2025.head())


Datos 2011-2025:
        Año                  Artista  Máxima_posición
57400  2011               Bruno Mars                1
57401  2011               Katy Perry                1
57402  2011                    Ke$ha                1
57403  2011  Rihanna Featuring Drake                1
57404  2011                     P!nk                1


In [ ]:
import pandas as pd
import altair as alt

# 🔧 Desactivar límites y vegafusion
alt.data_transformers.disable_max_rows()
alt.data_transformers.enable('default')

# === 1. CARGAR Y PROCESAR DATOS ===
df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')

# Convertir columna 'date' a año
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['Año'] = df['date'].dt.year

# Eliminar columnas innecesarias
df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])

# Renombrar columnas
df = df.rename(columns={
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Eliminar columna 'Canción'
df = df.drop(columns=['Canción'])

# Filtrar canciones con posición máxima entre 1 y 10
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]

# Filtrar rango de años 2011–2025
df_2011_2025 = df_filtered[df_filtered['Año'].between(2011, 2025)]

# === 2. AGRUPAR DATOS ===
pivot_table_top = (
    df_2011_2025
    .groupby(['Artista', 'Año'])
    .size()
    .unstack(fill_value=0)
)

# === 3. TRANSFORMAR DATOS PARA ALTAIR ===
df_long = (
    pivot_table_top
    .reset_index()
    .melt(id_vars='Artista', var_name='Año', value_name='Cantidad')
)

df_long['Año'] = df_long['Año'].astype(str)

# Mostrar solo los 20 artistas más frecuentes
top_artistas = (
    df_long.groupby('Artista')['Cantidad']
    .sum()
    .nlargest(20)
    .index
)
df_long = df_long[df_long['Artista'].isin(top_artistas)]

# === 4. CREAR VISUALIZACIÓN ===

# Selección para zoom
brush = alt.selection_interval(encodings=['x', 'y'])

# Selección para resaltar artista
highlight = alt.selection_point(fields=['Artista'], empty='none')

# --- Heatmap rosa con valores cero en blanco ---
heatmap = (
    alt.Chart(df_long)
    .mark_rect()
    .encode(
        x=alt.X('Año:O', title='Año'),
        y=alt.Y(
            'Artista:O',
            title='Artista',
            sort=alt.EncodingSortField(field='Cantidad', order='descending')
        ),
        color=alt.condition(
            alt.datum.Cantidad > 0,
            alt.Color('Cantidad:Q',
                      scale=alt.Scale(range=['#fde0dd', '#fa9fb5', '#c51b8a']),
                      title='Cantidad de canciones'),
            alt.value('white')  # ← Celdas sin valor quedan blancas
        ),
        tooltip=['Artista', 'Año', 'Cantidad']
    )
    .properties(
        width=700,
        height=500,
        title='Cantidad de entradas en Top 10 por artista y año (2011 - 2025)'
    )
    .add_params(brush, highlight)
    .transform_filter(brush)
)

# --- Texto al hacer clic ---
text = (
    alt.Chart(df_long)
    .mark_text(
        align='left',
        baseline='middle',
        dx=5,
        dy=-5,
        fontWeight='bold',
        color='black'
    )
    .encode(
        x='Año:O',
        y='Artista:O',
        text=alt.condition(highlight, 'Cantidad:Q', alt.value(''))
    )
    .add_params(highlight)
)

# --- Combinar ---
chart = (heatmap + text).configure(
    title=alt.TitleConfig(font='Poppins', fontSize=18, anchor='start', color='#c51b8a'),
    axis=alt.AxisConfig(labelFont='Poppins', titleFont='Poppins', labelFontSize=12, titleFontSize=13),
    legend=alt.LegendConfig(labelFont='Poppins', titleFont='Poppins'),
    view=alt.ViewConfig(stroke='transparent'),  # sin borde
    background='white'  # fondo blanco limpio
)

chart

alt.LayerChart(...)